# Vehicle Classification
*Note: Sensitive credentials and git tokens have been programmatically sanitized from this notebook.*

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes import GaussianNB
from sklearn.base import BaseEstimator, ClassifierMixin
SEED = 42


In [ ]:
import os
if not os.path.exists("vehicle_train_10000.csv"):
    from google.colab import files
    files.upload()   # select vehicle_train_10000.csv, vehicle_test.csv, vehicle_sample_submission.csv
train = pd.read_csv("vehicle_train_10000.csv")
test  = pd.read_csv("vehicle_test.csv")
sub   = pd.read_csv("vehicle_sample_submission.csv")
print(train.shape, test.shape, sub.shape)


In [ ]:
FEATURES = [c for c in train.columns if c not in ("vehicle_id", "vehicle_class")]
print("duplicate rows (ignoring id):", train.duplicated(subset=FEATURES).sum())
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)
X, y = df[FEATURES].values, df["vehicle_class"].values
X_test = test[FEATURES].values
print("after dedup:", X.shape, "| class balance:", np.bincount(y) / len(y))
print("missing values:", df.isna().sum().sum(), test.isna().sum().sum())


In [ ]:
class RegularizedLeastSquares(BaseEstimator, ClassifierMixin):
    """Ridge / Lasso regression on one-hot targets, predict argmax."""
    def __init__(self, kind="ridge", alpha=1.0): self.kind, self.alpha = kind, alpha
    def fit(self, X, y):
        self.classes_ = np.unique(y); Y = (y[:, None] == self.classes_).astype(float)
        reg = Ridge(alpha=self.alpha) if self.kind == "ridge" else Lasso(alpha=self.alpha)
        self.m_ = reg.fit(X, Y); return self
    def predict(self, X): return self.classes_[self.m_.predict(X).argmax(1)]

S = lambda m: make_pipeline(StandardScaler(), m)
CANDIDATES = {
    "Ridge least squares":   lambda: S(RegularizedLeastSquares("ridge", 10)),
    "Lasso least squares":   lambda: S(RegularizedLeastSquares("lasso", 0.01)),
    "Gaussian Naive Bayes":  lambda: S(GaussianNB()),
    "LDA":                   lambda: S(LDA()),
    "QDA (reg=0.1)":         lambda: S(QDA(reg_param=0.1)),
    "QDA (reg=0.3)":         lambda: S(QDA(reg_param=0.3)),
    "Softmax L1 (C=0.1)":    lambda: S(LogisticRegression(penalty="l1", solver="saga", C=0.1, max_iter=5000)),
    "Softmax L2 (C=1)":      lambda: S(LogisticRegression(C=1, max_iter=3000)),
    "Softmax L2 (C=0.1)":    lambda: S(LogisticRegression(C=0.1, max_iter=3000)),
    "Softmax L2 (C=0.01)":   lambda: S(LogisticRegression(C=0.01, max_iter=3000)),
}
NOISE_LEVELS = [0, 0.15, 0.3, 0.5]   # noise sd as a fraction of each feature's sd
sd = X.std(0)

def fit_with_aug(model, Xtr, ytr, aug, rng, copies=5):
    """Train on clean rows + noisy copies (aug=0 -> plain training)."""
    if aug > 0:
        Xtr = np.vstack([Xtr] + [Xtr + rng.normal(size=Xtr.shape) * sd * aug for _ in range(copies)])
        ytr = np.tile(ytr, copies + 1)
    return model.fit(Xtr, ytr)

def cv_scores(make_model, aug=0.0, n_splits=5, seed=SEED):
    rng = np.random.default_rng(seed); acc = np.zeros(len(NOISE_LEVELS))
    for tr, va in StratifiedKFold(n_splits, shuffle=True, random_state=seed).split(X, y):
        m = fit_with_aug(make_model(), X[tr], y[tr], aug, rng)
        for j, k in enumerate(NOISE_LEVELS):
            acc[j] += (m.predict(X[va] + rng.normal(size=X[va].shape) * sd * k) == y[va]).mean() / n_splits
    return acc


In [ ]:
rows = {name: cv_scores(mk) for name, mk in CANDIDATES.items()}
res = pd.DataFrame(rows, index=[f"val_noise={k}" for k in NOISE_LEVELS]).T
res["mean_over_noise"] = res.mean(axis=1)
pd.set_option("display.width", 200)
print(res.sort_values("mean_over_noise", ascending=False).round(4))


In [ ]:
grid = []
for C in [0.03, 0.1, 0.3, 1]:
    for aug in [0, 0.15, 0.3, 0.5]:
        a = cv_scores(lambda C=C: S(LogisticRegression(C=C, max_iter=3000)), aug=aug)
        grid.append([C, aug, *a, a.mean()])
tune = pd.DataFrame(grid, columns=["C", "train_noise"] + [f"val_noise={k}" for k in NOISE_LEVELS] + ["mean"])
print(tune.sort_values("mean", ascending=False).round(4).head(8))
best = tune.sort_values("mean", ascending=False).iloc[0]
BEST_C, BEST_AUG = float(best["C"]), float(best["train_noise"])
print("chosen C, train_noise:", BEST_C, BEST_AUG)   # expected: 0.1, 0.3


In [ ]:
final = fit_with_aug(S(LogisticRegression(C=BEST_C, max_iter=3000)), X, y, BEST_AUG, np.random.default_rng(SEED))
pred = final.predict(X_test)
submission = pd.DataFrame({"vehicle_id": test["vehicle_id"], "vehicle_class": pred})
assert list(submission.vehicle_id) == list(sub.vehicle_id)
submission.to_csv("submission.csv", index=False)
print(submission.vehicle_class.value_counts(normalize=True).round(3))
try:
    from google.colab import files; files.download("submission.csv")
except ImportError:
    pass


In [ ]:
!git config --global user.email "parampandey11@gmail.com"


In [ ]:
!git config --global user.email "parampandey11@gmail.com"
!git config --global user.name "halconie"


In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes import GaussianNB
from sklearn.base import BaseEstimator, ClassifierMixin
SEED = 42
